In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("100_Unique_QA_Dataset.csv")

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
# tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace("?", "")
    text = text.replace(".", "")
    return text.split()


In [3]:
tokenize("What is the largest planet in our solar system?")

['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system']

In [4]:
# vocab
vocab = {"<UNK>":0}

In [5]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [6]:
df.apply(build_vocab, axis = 1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [7]:
len(vocab)

326

In [8]:
# convert words to numerical indices
def text_to_indices(text, vacab):
    indexed_text = []
    
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab["<UNK>"])
    
    return indexed_text
            

In [9]:
print(text_to_indices("what is vinodCodesAi", vocab))

[1, 2, 0]


In [10]:
vocab["what"], vocab["is"] #, vocab["vinodCodesAi"]

(1, 2)

**Building RNN model Using PyTorch**

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

In [12]:
class QADataset(Dataset):
    
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
    
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        
        numerical_question = text_to_indices(self.df.iloc[index]["question"], self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]["answer"], self.vocab)
        
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [13]:
dataset = QADataset(df, vocab)

In [14]:
dataloader = DataLoader(dataset, batch_size = 1, shuffle = True)

In [15]:
for question, answer in dataloader:
    print(question, answer[0])

tensor([[ 10,  75, 111]]) tensor([112])
tensor([[42, 18,  2, 62, 63,  3, 64, 18]]) tensor([65])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([49])
tensor([[  1,   2,   3,  69,   5, 156]]) tensor([157])
tensor([[  1,   2,   3, 141, 117,  83,   3, 279, 280]]) tensor([121])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([116])
tensor([[ 10,  11, 190, 159, 191]]) tensor([192])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[10,  2,  3, 66,  5, 67]]) tensor([68])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[ 42, 292, 293, 118, 294, 159, 295, 296]]) tensor([297])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor([72])
tensor([[ 42,   2,   3, 276, 212, 277]]) tensor([278])
tensor([[  1,   2,   3,   4,   5, 207]]) tensor([208])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([134])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([16])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([36])
tensor([[  1,   2,   3, 147, 148,  19, 149]]) tensor([150])
tensor([[ 42, 101,   2,   3, 

In [16]:
import torch.nn as nn

In [23]:
class SimpleRNN(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim = 50)
        self.rnn = nn.RNN(50, 64, batch_first = True)
        self.fc = nn.Linear(64, vocab_size)
        
    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))
        
        return output


In [24]:
x = nn.Embedding(326, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 326)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 326])


In [25]:
# parameters
learning_rate = 0.001
epochs = 30


In [26]:
# model building
model = SimpleRNN(len(vocab))

In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [28]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.595761
Epoch: 2, Loss: 462.812463
Epoch: 3, Loss: 390.080145
Epoch: 4, Loss: 322.727899
Epoch: 5, Loss: 268.748661
Epoch: 6, Loss: 218.787567
Epoch: 7, Loss: 173.236243
Epoch: 8, Loss: 133.579364
Epoch: 9, Loss: 101.546693
Epoch: 10, Loss: 77.046011
Epoch: 11, Loss: 58.731898
Epoch: 12, Loss: 45.602581
Epoch: 13, Loss: 36.211876
Epoch: 14, Loss: 29.260286
Epoch: 15, Loss: 24.181884
Epoch: 16, Loss: 20.352683
Epoch: 17, Loss: 17.201794
Epoch: 18, Loss: 14.768281
Epoch: 19, Loss: 12.771285
Epoch: 20, Loss: 11.149708
Epoch: 21, Loss: 9.768827
Epoch: 22, Loss: 8.553130
Epoch: 23, Loss: 7.599120
Epoch: 24, Loss: 6.796065
Epoch: 25, Loss: 6.084774
Epoch: 26, Loss: 5.529801
Epoch: 27, Loss: 4.951130
Epoch: 28, Loss: 4.496006
Epoch: 29, Loss: 4.083777
Epoch: 30, Loss: 3.738718


In [29]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [30]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [31]:
list(vocab.keys())[7]

'paris'

In [32]:
predict(model, "What is vinodCodes-Ai.")

I don't know
parrot
